# Notebook 6: MDM + RAG — Document Intelligence with Entity Resolution

**Pattern:** Master Data Management (entity resolution) + Retrieval-Augmented Generation (document intelligence). Structured data meets unstructured documents meets AI.

**Stack:** DuckDB (entity resolution queries) + Docling (PDF parsing) + sentence-transformers (embeddings) + OpenSearch (vector search)

**Maple Trust Bank** — synthetic BFSI data

---

*Every previous notebook deferred Q2: "Find customers whose policy documents reference AML procedure X." This one answers it — by combining policy document intelligence (RAG) with entity resolution (MDM).*

---
## Section 1: The Pattern in One Paragraph

**Master Data Management** solves the identity problem: when a customer appears as `CUST-000001` in core banking, `CL-MTB-000001` in CRM, and `ENT-000001` in the AML system, entity resolution produces a single golden record that links all three. **Retrieval-Augmented Generation** solves the knowledge problem: the bank's 10 compliance policy PDFs — AML, KYC, sanctions, EDD — become searchable via embeddings and vector search, so a compliance officer can ask "what procedures apply to high-risk customers?" and get an answer grounded in the actual policy text. Together: MDM tells you *who* (which customers, across all source systems), RAG tells you *what* (which policy rules apply), and the combination answers questions that no single pattern from Notebooks 1–5 could. This is not a customer-to-document lookup — there is no per-customer document index. It is **policy-driven customer identification**: discover the rule from the documents, then find the people the rule applies to.

---
## Section 2: When You'd Use It, When You Wouldn't

| Use when | Don't use when |
|----------|----------------|
| Cross-system identity resolution needed (different IDs per source system) | Single source system with one canonical ID |
| Unstructured policy/regulatory documents must inform structured-data decisions | All queries are pure SQL over structured tables |
| Compliance requires linking actions to source policy sections (audit trail) | Simple keyword search over documents suffices |
| Customer 360 view spanning multiple systems for regulatory reporting | Each system operates independently |
| AI-assisted Q&A over a corpus of internal policy documents | Documents are already tagged and indexed by humans |
| You need to answer "which customers does this policy apply to?" | You already know the answer and just need to query it |

---
## Section 3: The Setup

Three phases:
1. **Load MDM entity links** — pre-computed entity resolution across 4 source systems
2. **Parse policy PDFs + generate embeddings** — Docling for PDF-to-text, sentence-transformers for vectors
3. **Index into OpenSearch** — knn vector search over policy document chunks

Plan A: IBM Match 360 (entity resolution) + Watson Discovery (document intelligence). Plan B (this notebook): pre-computed `entity_links.parquet` + Docling + sentence-transformers + OpenSearch.

> Only OpenSearch is required for this notebook. If other Docker services aren't needed: `docker compose up -d opensearch`

In [1]:
# ── Configuration + Imports ────────────────────────────────────────────
PLAN = "B"  # "A" for IBM Match 360 + Watson Discovery; "B" for local stack

OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX = "nb6_policy_docs"  # notebook-specific index name

EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # 384-dim, fast
# NOTE: First run downloads ~80MB from HuggingFace. Ensure internet access.
# NOTE: Pin OpenSearch image in docker-compose.yml for reproducibility
#       (e.g., opensearchproject/opensearch:2.11.1 instead of :2)

DATA_DIR = "../data"

import pandas as pd
import duckdb
import json
import time
import re
from pathlib import Path
from opensearchpy import OpenSearch
from opensearchpy.helpers import bulk
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

if PLAN == "A":
    print("Plan A: IBM Match 360 + Watson Discovery (configure credentials above)")
else:
    print("Plan B: Local entity_links.parquet + Docling + sentence-transformers + OpenSearch")

Plan B: Local entity_links.parquet + Docling + sentence-transformers + OpenSearch


In [4]:
# ── Pre-flight checks ──────────────────────────────────────────────────
# 1. OpenSearch connectivity with retry
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    use_ssl=False,
)

for attempt in range(6):
    try:
        health = os_client.cluster.health()
        print(f"OpenSearch: {health['cluster_name']} (status: {health['status']})")
        break
    except Exception as e:
        if attempt < 5:
            print(f"  Waiting for OpenSearch... (attempt {attempt + 1}/6)")
            time.sleep(5)
        else:
            raise RuntimeError(
                f"OpenSearch not reachable at {OPENSEARCH_HOST}:{OPENSEARCH_PORT}. "
                "Run: docker compose up -d opensearch"
            ) from e

# 2. Embedding model check
print(f"Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)
dim = model.get_embedding_dimension()
print(f"  Dimensions: {dim}")

# 3. DuckDB
con = duckdb.connect()
print("DuckDB: connected (in-memory)")

OpenSearch: docker-cluster (status: green)
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Dimensions: 384
DuckDB: connected (in-memory)


In [5]:
# ── Phase 1: Load MDM entity links + core data ─────────────────────────
data_dir = Path(DATA_DIR)

entity_links = pd.read_parquet(data_dir / "mdm" / "entity_links.parquet")
customers = pd.read_parquet(data_dir / "customers.parquet")
accounts = pd.read_parquet(data_dir / "accounts.parquet")
transactions = pd.read_parquet(data_dir / "transactions.parquet")
branches = pd.read_parquet(data_dir / "branches.parquet")

con.register("entity_links", entity_links)
con.register("customers", customers)
con.register("accounts", accounts)
con.register("transactions", transactions)
con.register("branches", branches)

n_entities = entity_links["entity_id"].nunique()
print(f"Loaded: {len(entity_links)} entity links ({n_entities} unique entities)")
print(f"  Source systems: {sorted(entity_links['source_system'].unique().tolist())}")
print(f"  Match methods:  {sorted(entity_links['match_method'].unique().tolist())}")
print(f"  Confidence range: {entity_links['match_confidence'].min():.3f} – {entity_links['match_confidence'].max():.3f}")
print(f"\nCore data: customers={len(customers):,}, accounts={len(accounts):,}, "
      f"transactions={len(transactions):,}, branches={len(branches)}")

Loaded: 500 entity links (176 unique entities)
  Source systems: ['aml_system', 'branch_records', 'core_banking', 'crm']
  Match methods:  ['exact_name_dob', 'fuzzy_name_address', 'manual_review', 'ssn_match']
  Confidence range: 0.701 – 1.000

Core data: customers=100,000, accounts=200,000, transactions=1,000,000, branches=50


In [6]:
# ── The identity problem (bridging from NB5) ──────────────────────────
print("📊 Reference Architecture Swimlane: Information & Model Management & Governance")
print("   (IBM Match 360 — Master Data Management)\n")

print("NB5's cross-domain join worked because the simulation used the same")
print("customer_id everywhere. Real banks don't:")
print("  Retail Banking:    CUST-000001")
print("  CRM:               CL-MTB-000001")
print("  AML System:        ENT-000001")
print("  Branch Records:    BR-MTB-000001")
print()
print("Entity resolution links these different IDs to ONE golden record.\n")

# Show one entity resolved across multiple source systems
sample = con.execute("""
    SELECT el.entity_id, el.source_system, el.source_id, el.customer_id,
           el.match_confidence, el.match_method, c.name
    FROM entity_links el
    JOIN customers c ON el.customer_id = c.customer_id
    WHERE el.entity_id = (
        SELECT entity_id FROM entity_links
        GROUP BY entity_id
        HAVING COUNT(DISTINCT source_system) = 4
        LIMIT 1
    )
    ORDER BY el.source_system
""").fetchdf()

print(f"Example: entity {sample['entity_id'].iloc[0]} — one person, {len(sample)} system identities:")
print(sample.to_string(index=False))

📊 Reference Architecture Swimlane: Information & Model Management & Governance
   (IBM Match 360 — Master Data Management)

NB5's cross-domain join worked because the simulation used the same
customer_id everywhere. Real banks don't:
  Retail Banking:    CUST-000001
  CRM:               CL-MTB-000001
  AML System:        ENT-000001
  Branch Records:    BR-MTB-000001

Entity resolution links these different IDs to ONE golden record.

Example: entity ENT-0150 — one person, 4 system identities:
entity_id  source_system  source_id customer_id  match_confidence       match_method         name
 ENT-0150     aml_system  AML-21153 CUST-040172             0.849 fuzzy_name_address Wayne Porter
 ENT-0150 branch_records   BR-77775 CUST-040172             0.953      manual_review Wayne Porter
 ENT-0150   core_banking  CB-600289 CUST-040172             0.966     exact_name_dob Wayne Porter
 ENT-0150            crm CRM-713705 CUST-040172             0.879      manual_review Wayne Porter


In [7]:
# ── Phase 2: Parse policy PDFs with Docling ───────────────────────────
# NOTE: This cell takes 30-60 seconds. Docling does deep PDF layout analysis.
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
policy_dir = data_dir / "policies"
pdf_files = sorted(policy_dir.glob("MTB-POL-*.pdf"))

parsed_docs = {}
parse_errors = []

for pdf_path in pdf_files:
    try:
        result = converter.convert(str(pdf_path))
        md_text = result.document.export_to_markdown()
        if md_text.strip():
            parsed_docs[pdf_path.name] = md_text
            # Count sections (## headings)
            sections = len(re.findall(r'^#{1,3} ', md_text, re.MULTILINE))
            print(f"  ✓ {pdf_path.name}: {len(md_text):,} chars, {sections} sections")
        else:
            parse_errors.append(pdf_path.name)
            print(f"  ⚠️ {pdf_path.name}: parsed but empty text")
    except Exception as e:
        parse_errors.append(pdf_path.name)
        print(f"  ✗ {pdf_path.name}: {e}")

print(f"\n✓ Parsed {len(parsed_docs)}/{len(pdf_files)} policy documents")
if parse_errors:
    print(f"  ⚠️ Failed: {parse_errors}")

[INFO] 2026-05-07 09:49:13,049 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-07 09:49:13,053 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-07 09:49:13,062 [RapidOCR] download_file.py:60: File exists and is valid: /Users/mg/mg-work/manav/work/ai-experiments/data-architecture/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-07 09:49:13,062 [RapidOCR] main.py:50: Using /Users/mg/mg-work/manav/work/ai-experiments/data-architecture/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-07 09:49:13,158 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-07 09:49:13,158 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-07 09:49:13,160 [RapidOCR] download_file.py:60: File exists and is valid: /Users/mg/mg-work/manav/work/ai-experiments/data-architecture/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  ✓ MTB-POL-001.pdf: 43,956 chars, 45 sections
  ✓ MTB-POL-002.pdf: 29,857 chars, 35 sections
  ✓ MTB-POL-003.pdf: 27,431 chars, 33 sections
  ✓ MTB-POL-004.pdf: 28,848 chars, 32 sections
  ✓ MTB-POL-005.pdf: 22,877 chars, 25 sections
  ✓ MTB-POL-006.pdf: 32,280 chars, 39 sections
  ✓ MTB-POL-007.pdf: 22,546 chars, 25 sections
  ✓ MTB-POL-008.pdf: 20,788 chars, 20 sections
  ✓ MTB-POL-009.pdf: 26,823 chars, 27 sections
  ✓ MTB-POL-010.pdf: 28,862 chars, 27 sections

✓ Parsed 10/10 policy documents


In [8]:
# ── Section-aware chunking ─────────────────────────────────────────────
MAX_CHUNK_CHARS = 1024
SUB_CHUNK_SIZE = 512
SUB_CHUNK_OVERLAP = 64


def chunk_document(doc_name, md_text):
    """Split markdown into section-aware chunks.
    
    Primary split: on ## and ### headings (from Docling output).
    Secondary split: char-based for sections exceeding MAX_CHUNK_CHARS.
    """
    # Split on markdown headings
    sections = re.split(r'(?=^#{1,3} )', md_text, flags=re.MULTILINE)
    sections = [s for s in sections if s.strip()]

    chunks = []
    for section in sections:
        # Extract heading for metadata
        heading_match = re.match(r'^(#{1,3}) (.+)', section)
        heading = heading_match.group(2).strip() if heading_match else "(no heading)"

        if len(section) <= MAX_CHUNK_CHARS:
            chunks.append({
                "doc_name": doc_name,
                "section_heading": heading,
                "chunk_id": f"{doc_name}::chunk-{len(chunks):03d}",
                "text": section.strip(),
            })
        else:
            # Sub-split large sections
            start = 0
            sub_idx = 0
            while start < len(section):
                end = start + SUB_CHUNK_SIZE
                chunks.append({
                    "doc_name": doc_name,
                    "section_heading": f"{heading} (part {sub_idx + 1})",
                    "chunk_id": f"{doc_name}::chunk-{len(chunks):03d}",
                    "text": section[start:end].strip(),
                })
                start += SUB_CHUNK_SIZE - SUB_CHUNK_OVERLAP
                sub_idx += 1
    return chunks


all_chunks = []
for doc_name, md_text in parsed_docs.items():
    doc_chunks = chunk_document(doc_name, md_text)
    all_chunks.extend(doc_chunks)

print(f"✓ Created {len(all_chunks)} chunks from {len(parsed_docs)} documents")
print(f"  Section-aware primary split, sub-split at {SUB_CHUNK_SIZE} chars / {SUB_CHUNK_OVERLAP} overlap")
print(f"  Avg chunk size: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} chars")

✓ Created 637 chunks from 10 documents
  Section-aware primary split, sub-split at 512 chars / 64 overlap
  Avg chunk size: 473 chars


In [9]:
# ── Generate embeddings ────────────────────────────────────────────────
texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

assert embeddings.shape[0] == len(all_chunks), (
    f"Embedding count {embeddings.shape[0]} != chunk count {len(all_chunks)}"
)
print(f"\n✓ Generated {len(embeddings)} embeddings")
print(f"  Dimensions: {embeddings.shape[1]}, dtype: {embeddings.dtype}")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


✓ Generated 637 embeddings
  Dimensions: 384, dtype: float32


In [10]:
# ── Index into OpenSearch ──────────────────────────────────────────────
# Create index if it doesn't exist (notebook-specific name)
if os_client.indices.exists(index=OPENSEARCH_INDEX):
    print(f"Index '{OPENSEARCH_INDEX}' already exists — deleting for fresh run.")
    os_client.indices.delete(index=OPENSEARCH_INDEX)

index_body = {
    "settings": {
        "index": {"knn": True, "number_of_replicas": 0}
    },
    "mappings": {
        "properties": {
            "embedding": {
                "type": "knn_vector",
                "dimension": embeddings.shape[1],
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "lucene",
                }
            },
            "text": {"type": "text"},
            "doc_name": {"type": "keyword"},
            "section_heading": {"type": "keyword"},
            "chunk_id": {"type": "keyword"},
        }
    }
}

os_client.indices.create(index=OPENSEARCH_INDEX, body=index_body)

# Bulk index
actions = []
for i, chunk in enumerate(all_chunks):
    actions.append({
        "_index": OPENSEARCH_INDEX,
        "_id": chunk["chunk_id"],
        "_source": {
            "text": chunk["text"],
            "doc_name": chunk["doc_name"],
            "section_heading": chunk["section_heading"],
            "chunk_id": chunk["chunk_id"],
            "embedding": embeddings[i].tolist(),
        }
    })

success, errors = bulk(os_client, actions)
os_client.indices.refresh(index=OPENSEARCH_INDEX)

print(f"✓ Indexed {success} chunks into OpenSearch index '{OPENSEARCH_INDEX}'")
if errors:
    print(f"  ⚠️ {len(errors)} indexing errors")
    for err in errors[:3]:
        print(f"    {err}")

Index 'nb6_policy_docs' already exists — deleting for fresh run.
✓ Indexed 637 chunks into OpenSearch index 'nb6_policy_docs'


---
## Section 4: Three Canonical Queries

The same three business questions from Notebooks 1–5. This time, we have entity resolution AND document intelligence.

> **Reference Architecture Swimlane:** AI & Model Management (RAG pipeline) + Information & Model Management & Governance (MDM entity resolution)

In [11]:
# ── Q1: Entity-resolved customer_360 view ─────────────────────────────
# 🎯 Cameo cell — run this one during Block 6 for a quick demo.
print("📊 Reference Architecture Swimlane: Information & Model Management & Governance")
print("   (IBM Match 360 — entity resolution)\n")

# How many entities span multiple source systems?
system_counts = con.execute("""
    SELECT
        COUNT(DISTINCT source_system) AS n_systems,
        COUNT(DISTINCT entity_id) AS n_entities
    FROM entity_links
    GROUP BY entity_id
""").fetchdf()

dist = system_counts.groupby("n_systems")["n_entities"].sum()
print("Entity resolution coverage:")
for n_sys, count in sorted(dist.items()):
    print(f"  {n_sys} source system(s): {count} entities")

print()

# Show a sample entity with all its identities
sample_entity = con.execute("""
    SELECT el.entity_id, el.source_system, el.source_id,
           el.match_confidence, c.name, c.risk_score
    FROM entity_links el
    JOIN customers c ON el.customer_id = c.customer_id
    WHERE el.entity_id = (
        SELECT entity_id FROM entity_links
        GROUP BY entity_id
        HAVING COUNT(DISTINCT source_system) = 4
        ORDER BY entity_id LIMIT 1
    )
    ORDER BY el.source_system
""").fetchdf()

print(f"Sample: {sample_entity['entity_id'].iloc[0]} — {sample_entity['name'].iloc[0]}")
print(sample_entity[["source_system", "source_id", "match_confidence"]].to_string(index=False))
print()
print("**Key insight:** Without MDM, this customer appears as 4 different records.")
print("Regulatory reports (OSFI, FINTRAC) require consolidated identity.")

📊 Reference Architecture Swimlane: Information & Model Management & Governance
   (IBM Match 360 — entity resolution)

Entity resolution coverage:
  2 source system(s): 61 entities
  3 source system(s): 82 entities
  4 source system(s): 33 entities

Sample: ENT-0009 — Erin Johnson
 source_system  source_id  match_confidence
    aml_system  AML-47519             0.968
branch_records   BR-95396             0.891
  core_banking  CB-675606             0.957
           crm CRM-394745             0.984

**Key insight:** Without MDM, this customer appears as 4 different records.
Regulatory reports (OSFI, FINTRAC) require consolidated identity.


### Q2: Find customers whose policy documents reference AML procedure X

**This is the question every previous notebook deferred.**

Here's how we answer it:
1. **RAG** retrieves the policy sections that define AML procedures for high-risk customers
2. We extract the customer criteria from those sections (e.g., "risk_score >= 71 triggers EDD")
3. **MDM** + structured query finds matching customers across all source systems

> **Cameo cell** — run the next two cells during Block 6 for a quick demo.

In [12]:
# ── Q2 Step 1: RAG retrieval — search policy documents ────────────────
# 🎯 Cameo cell — the payoff of the entire lecture series.
print("📊 Reference Architecture Swimlane: AI & Model Management")
print("   (Watson Discovery / RAG pipeline)\n")

query = "Enhanced Due Diligence procedures for high-risk customers under AML policy"
query_embedding = model.encode([query])[0].tolist()

search_body = {
    "size": 5,
    "query": {
        "knn": {
            "embedding": {
                "vector": query_embedding,
                "k": 5,
            }
        }
    },
    "_source": ["text", "doc_name", "section_heading", "chunk_id"],
}

response = os_client.search(index=OPENSEARCH_INDEX, body=search_body)

print(f'Query: "{query}"\n')
print(f"Top {len(response['hits']['hits'])} retrieved chunks:")
print("=" * 70)
for i, hit in enumerate(response["hits"]["hits"]):
    src = hit["_source"]
    print(f"\n  [{i+1}] Score: {hit['_score']:.4f}")
    print(f"      Source: {src['doc_name']} — {src['section_heading']}")
    text_preview = src["text"][:200].replace("\n", " ")
    print(f"      Text:  {text_preview}...")

📊 Reference Architecture Swimlane: AI & Model Management
   (Watson Discovery / RAG pipeline)

Query: "Enhanced Due Diligence procedures for high-risk customers under AML policy"

Top 5 retrieved chunks:

  [1] Score: 0.8505
      Source: MTB-POL-001.pdf — 6. Customer Due Diligence
      Text:  ## 6. Customer Due Diligence  Customer Due Diligence (CDD) is the cornerstone of Maple Trust Bank's AML program. Detailed CDD procedures are defined in MTB-POL-004.  - Identity verification at account...

  [2] Score: 0.8109
      Source: MTB-POL-001.pdf — 16.2 Risk Committee Responsibilities
      Text:  ## 16.2 Risk Committee Responsibilities  - Quarterly review of AML program performance metrics and KRIs - Review and approval of changes to monitoring scenarios and thresholds - Oversight of remediati...

  [3] Score: 0.8017
      Source: MTB-POL-005.pdf — 1. Purpose
      Text:  ## 1. Purpose  This document outlines the Enhanced Due Diligence (EDD) standards for high-risk customers at Maple Tr

In [13]:
# ── Q2 Step 2: Extract criteria + find matching customers ─────────────
print("Policy finding: EDD (MTB-POL-005) applies to customers with elevated risk.")
print("The policy defines tiers — we query for the highest-risk entity-resolved customers.\n")

q2_results = con.execute("""
    SELECT
        el.entity_id,
        c.customer_id,
        c.name,
        c.risk_score,
        c.kyc_status,
        c.segment,
        COUNT(DISTINCT el.source_system) AS n_source_systems,
        MIN(el.match_confidence) AS min_confidence
    FROM customers c
    JOIN entity_links el ON c.customer_id = el.customer_id
    WHERE c.risk_score >= 40
      AND el.match_confidence >= 0.90
    GROUP BY el.entity_id, c.customer_id, c.name, c.risk_score,
             c.kyc_status, c.segment
    ORDER BY c.risk_score DESC
""").fetchdf()

n_unique = q2_results["entity_id"].nunique()
print(f"Q2 ANSWERED: {n_unique} unique entities flagged for enhanced review")
print(f"  (risk_score >= 40, entity confidence >= 0.90)\n")
q2_results.head(10)

Policy finding: EDD (MTB-POL-005) applies to customers with elevated risk.
The policy defines tiers — we query for the highest-risk entity-resolved customers.

Q2 ANSWERED: 11 unique entities flagged for enhanced review
  (risk_score >= 40, entity confidence >= 0.90)



,entity_id,customer_id,name,risk_score,kyc_status,segment,n_source_systems,min_confidence
0,ENT-0045,CUST-012623,Joseph Morris,58,verified,retail,4,0.964
1,ENT-0054,CUST-069660,Nathaniel Marks,52,verified,wealth,3,0.992
2,ENT-0138,CUST-075689,Joshua Calderon,52,verified,retail,3,0.952
3,ENT-0116,CUST-095965,Lacey Wilson,49,verified,retail,1,0.992
4,ENT-0064,CUST-083172,Travis Trevino,47,verified,wealth,2,0.984
5,ENT-0145,CUST-049946,Matthew Lopez,44,verified,retail,1,0.960
6,ENT-0100,CUST-085690,Donald Schneider,43,verified,retail,3,0.975
7,ENT-0027,CUST-016620,Raymond Murray,41,verified,retail,2,0.945
8,ENT-0083,CUST-043646,Richard Villanueva,40,verified,retail,1,0.960
9,ENT-0143,CUST-055680,George Adams,40,verified,institutional,2,0.986


**What just happened:**

1. **RAG** searched 10 policy PDFs and found that MTB-POL-005 (Enhanced Due Diligence) defines tiered procedures based on customer risk scores
2. We extracted the criterion: customers with elevated risk scores require enhanced review
3. **MDM** identified all unique entities matching that criterion — across core_banking, CRM, AML system, and branch records — with entity resolution confidence >= 0.90

This is **policy-driven customer identification**: the bank's compliance policies become queryable knowledge that drives structured-data actions.

> **Key insight:** NB1–NB5 could query the structured data but couldn't read the policies. The warehouse stored risk scores but didn't know *why* a particular score matters. The lake stored the PDFs but couldn't query them. The lakehouse could store embeddings but had no RAG pipeline. Virtualization could federate the structured data but not the documents. The mesh showed the identity problem but couldn't resolve it. NB6 closes the loop.

In [14]:
# ── Q3: Lineage — entity resolution + RAG pipeline ────────────────────
print("📊 Reference Architecture Swimlane: Discovery & Exploration")
print("   (IBM Knowledge Catalog — data lineage)\n")

with open(data_dir / "lineage" / "lineage_graph.json") as f:
    lineage = json.load(f)

# Trace real lineage edges involving customer_360
target = "curated.customer_360"
print(f"Lineage for: {target}")
print("=" * 60)

# Upstream (what feeds into customer_360)
upstream = [e for e in lineage["edges"] if e["to"] == target]
print(f"\nUpstream ({len(upstream)} sources):")
for e in upstream:
    print(f"  ← {e['from']:40s} transform: {e['transform']}")

# Downstream (what customer_360 feeds into)
downstream = [e for e in lineage["edges"] if e["from"] == target]
print(f"\nDownstream ({len(downstream)} consumers):")
for e in downstream:
    print(f"  → {e['to']:40s} transform: {e['transform']}")

print("\n" + "=" * 60)
print("\nThe RAG pipeline is NOT in this lineage graph — it's new.")
print("In production, IBM Knowledge Catalog would track it as:")
print("  data/policies/*.pdf → Docling parse → sentence-transformers embed")
print(f"  → OpenSearch index '{OPENSEARCH_INDEX}' → Q2 retrieval")
print("  → customer identification via entity_links join")

📊 Reference Architecture Swimlane: Discovery & Exploration
   (IBM Knowledge Catalog — data lineage)

Lineage for: curated.customer_360

Upstream (1 sources):
  ← raw.customers                            transform: entity resolution + dedup + PII standardization + address normalization

Downstream (5 consumers):
  → curated.accounts_enriched                transform: customer segment and risk profile join
  → consumed.aml_alerts                      transform: customer risk profile enrichment for alert context
  → consumed.customer_risk_scores            transform: weighted risk factor scoring model (MTB-POL-010 methodology)
  → consumed.customer_segmentation           transform: feature engineering + k-means clustering + segment labelling
  → consumed.kyc_dashboard_data              transform: customer segment dimensions for KYC metric drill-down


The RAG pipeline is NOT in this lineage graph — it's new.
In production, IBM Knowledge Catalog would track it as:
  data/policies/*.pdf → 

In [15]:
# ── Q3 continued: RAG provenance ──────────────────────────────────────
print("RAG provenance for Q2 answer:")
print("=" * 60)
print()

for i, hit in enumerate(response["hits"]["hits"]):
    src = hit["_source"]
    print(f"  Source {i+1}: {src['doc_name']}")
    print(f"    Section: {src['section_heading']}")
    print(f"    Score:   {hit['_score']:.4f}")
    print()

print("This provenance trail is critical for regulatory compliance:")
print("  → OSFI requires audit trails for risk-related decisions")
print("  → The bank must prove: 'we applied EDD because MTB-POL-005 says so'")
print("  → RAG provenance links the decision to the specific policy section")

RAG provenance for Q2 answer:

  Source 1: MTB-POL-001.pdf
    Section: 6. Customer Due Diligence
    Score:   0.8505

  Source 2: MTB-POL-001.pdf
    Section: 16.2 Risk Committee Responsibilities
    Score:   0.8109

  Source 3: MTB-POL-005.pdf
    Section: 1. Purpose
    Score:   0.8017

  Source 4: MTB-POL-010.pdf
    Section: 12. Third-Party Risk Assessment (part 1)
    Score:   0.8012

  Source 5: MTB-POL-001.pdf
    Section: Table of Contents (part 1)
    Score:   0.7960

This provenance trail is critical for regulatory compliance:
  → OSFI requires audit trails for risk-related decisions
  → The bank must prove: 'we applied EDD because MTB-POL-005 says so'
  → RAG provenance links the decision to the specific policy section


---
## Section 5: Where This Pattern Breaks

In [16]:
# ── Break 1: Low-confidence entity matches ────────────────────────────
print("Break 1: Entity resolution quality")
print("=" * 60)
print()

confidence_stats = con.execute("""
    SELECT match_method,
           COUNT(*) AS n_links,
           ROUND(MIN(match_confidence), 3) AS min_conf,
           ROUND(AVG(match_confidence), 3) AS avg_conf,
           ROUND(MAX(match_confidence), 3) AS max_conf,
           SUM(CASE WHEN match_confidence < 0.80 THEN 1 ELSE 0 END) AS below_80
    FROM entity_links
    GROUP BY match_method
    ORDER BY avg_conf
""").fetchdf()

print("Match confidence by method:")
print(confidence_stats.to_string(index=False))
print()

low_conf_count = int((entity_links["match_confidence"] < 0.80).sum())
low_conf_entities = entity_links[entity_links["match_confidence"] < 0.80]["entity_id"].nunique()
print(f"Links below 0.80 confidence: {low_conf_count}")
print(f"Entities with at least one risky link: {low_conf_entities}")
print()
print("⚠️  These could be WRONG — linking two different people as the same entity.")
print("In production, IBM Match 360 queues low-confidence matches for human review.")
print("This notebook uses pre-computed links — it cannot demonstrate adjudication")
print("or survivorship rules. That's the gap between a demo and an enterprise system.")

Break 1: Entity resolution quality

Match confidence by method:
      match_method  n_links  min_conf  avg_conf  max_conf  below_80
fuzzy_name_address      132     0.701     0.831     0.949      45.0
     manual_review       77     0.802     0.884     0.985       0.0
    exact_name_dob      219     0.950     0.975     1.000       0.0
         ssn_match       72     0.980     0.991     0.999       0.0

Links below 0.80 confidence: 45
Entities with at least one risky link: 43

⚠️  These could be WRONG — linking two different people as the same entity.
In production, IBM Match 360 queues low-confidence matches for human review.
This notebook uses pre-computed links — it cannot demonstrate adjudication
or survivorship rules. That's the gap between a demo and an enterprise system.


In [17]:
# ── Break 2: RAG retrieval quality — full eval ────────────────────────
print("Break 2: RAG retrieval quality on evaluation set")
print("=" * 60)
print()

eval_data = []
with open(data_dir / "eval" / "aml_qa_eval.jsonl") as f:
    for line in f:
        if line.strip():
            eval_data.append(json.loads(line))

print(f"Loaded {len(eval_data)} evaluation questions\n")

k = 5
results = []

for item in eval_data:
    q_emb = model.encode([item["question"]])[0].tolist()
    search_body = {
        "size": k,
        "query": {"knn": {"embedding": {"vector": q_emb, "k": k}}},
        "_source": ["doc_name"],
    }
    resp = os_client.search(index=OPENSEARCH_INDEX, body=search_body)
    retrieved_docs = [hit["_source"]["doc_name"] for hit in resp["hits"]["hits"]]

    expected_docs = [d.strip() for d in item["source_doc"].split(",")]
    is_multi = len(expected_docs) > 1
    hit = any(doc in retrieved_docs for doc in expected_docs)

    results.append({
        "question": item["question"][:70],
        "expected": item["source_doc"],
        "retrieved_top1": retrieved_docs[0] if retrieved_docs else "none",
        "hit": hit,
        "is_multi": is_multi,
    })

results_df = pd.DataFrame(results)

# Per-category recall
single_doc = results_df[~results_df["is_multi"]]
multi_doc = results_df[results_df["is_multi"]]

single_recall = single_doc["hit"].mean() if len(single_doc) > 0 else 0
multi_recall = multi_doc["hit"].mean() if len(multi_doc) > 0 else 0
overall_recall = results_df["hit"].mean()

print(f"Recall@{k} (at least one expected doc in top {k}):")
print(f"  Single-doc questions ({len(single_doc)}): {single_recall:.1%}")
print(f"  Multi-doc questions  ({len(multi_doc)}):  {multi_recall:.1%}")
print(f"  Overall              ({len(results_df)}): {overall_recall:.1%}")
print()

# Show worst failures
failures = results_df[~results_df["hit"]]
if len(failures) > 0:
    print(f"Failed retrievals ({len(failures)}):")
    for _, row in failures.head(5).iterrows():
        print(f"  ✗ Q: {row['question']}...")
        print(f"    Expected: {row['expected']}")
        print(f"    Got top-1: {row['retrieved_top1']}")
        print()
else:
    print("All questions retrieved at least one correct doc in top 5.")

Break 2: RAG retrieval quality on evaluation set

Loaded 30 evaluation questions

Recall@5 (at least one expected doc in top 5):
  Single-doc questions (17): 100.0%
  Multi-doc questions  (13):  92.3%
  Overall              (30): 96.7%

Failed retrievals (1):
  ✗ Q: What data quality requirements exist across the policy framework, and ...
    Expected: MTB-POL-009.pdf,MTB-POL-006.pdf,MTB-POL-002.pdf
    Got top-1: MTB-POL-010.pdf



In [18]:
# ── Break 3: Multi-doc questions lose cross-section context ────────────
print("Break 3: Multi-document reasoning failure")
print("=" * 60)
print()

# Pick a multi-doc question that spans 3+ policies
multi_doc_items = [item for item in eval_data
                   if len(item["source_doc"].split(",")) >= 3]

if multi_doc_items:
    test_item = multi_doc_items[0]
    expected = [d.strip() for d in test_item["source_doc"].split(",")]

    print(f'Question: "{test_item["question"][:100]}..."')
    print(f"Expected docs ({len(expected)}): {expected}\n")

    q_emb = model.encode([test_item["question"]])[0].tolist()
    search_body = {
        "size": 5,
        "query": {"knn": {"embedding": {"vector": q_emb, "k": 5}}},
        "_source": ["doc_name", "section_heading"],
    }
    resp = os_client.search(index=OPENSEARCH_INDEX, body=search_body)

    retrieved = set()
    for hit in resp["hits"]["hits"]:
        doc = hit["_source"]["doc_name"]
        section = hit["_source"]["section_heading"]
        retrieved.add(doc)
        in_expected = "✓" if doc in expected else "✗"
        print(f"  {in_expected} {doc} — {section} (score: {hit['_score']:.4f})")

    found = set(expected) & retrieved
    missed = set(expected) - retrieved
    print(f"\nFound {len(found)}/{len(expected)} expected docs: {sorted(found)}")
    if missed:
        print(f"Missed: {sorted(missed)}")

print()
print("Single-vector retrieval finds one or two relevant sections, not all.")
print("Production RAG needs: query decomposition, multi-hop retrieval, re-ranking.")
print("Watson Discovery handles this with domain-adapted models and passage ranking.")

Break 3: Multi-document reasoning failure

Question: "If a customer's risk score increases from 60 to 75, what additional due diligence requirements apply..."
Expected docs (3): ['MTB-POL-001.pdf', 'MTB-POL-002.pdf', 'MTB-POL-005.pdf']

  ✓ MTB-POL-001.pdf — 5.2 Risk Scoring Model (score: 0.7991)
  ✓ MTB-POL-002.pdf — 4. Customer Risk Assessment (score: 0.7928)
  ✓ MTB-POL-001.pdf — 5.1 Customer Risk Factors (score: 0.7804)
  ✗ MTB-POL-010.pdf — Table of Contents (score: 0.7685)
  ✗ MTB-POL-010.pdf — 5. Enterprise Risk Assessment Process (score: 0.7560)

Found 2/3 expected docs: ['MTB-POL-001.pdf', 'MTB-POL-002.pdf']
Missed: ['MTB-POL-005.pdf']

Single-vector retrieval finds one or two relevant sections, not all.
Production RAG needs: query decomposition, multi-hop retrieval, re-ranking.
Watson Discovery handles this with domain-adapted models and passage ranking.


**Summary of breaks:**

- **Entity resolution** quality depends on match method and confidence thresholds. Fuzzy name/address matching produces the lowest confidence and requires human review.
- **RAG retrieval** degrades on multi-document reasoning questions — single-vector search finds one relevant section, not all four policies that contribute to the answer.
- Both need **human oversight** in production. The technology surfaces candidates; people make the final call.

---
## Section 6: The IBM Stack Mapping

| Component | Plan B (Local) | Plan A (IBM Product) | Swimlane |
|-----------|---------------|---------------------|----------|
| Entity Resolution | entity_links.parquet (pre-computed) | IBM Match 360 | Information & Model Management & Governance |
| Document Parsing | Docling (IBM Research OSS) | Watson Discovery (managed) | AI & Model Management |
| Embeddings | sentence-transformers (all-MiniLM-L6-v2) | watsonx.ai embedding models | AI & Model Management |
| Vector Store | OpenSearch (self-managed) | Watson Discovery / Elasticsearch | Analytical Data Management & Storage |
| RAG Orchestration | Manual (Python) | watsonx.ai RAG pattern | AI & Model Management |
| Lineage | lineage_graph.json (static) | IBM Knowledge Catalog | Discovery & Exploration |
| Governance | None (demo) | Data Privacy Passports, AI Factsheets | Information & Model Management & Governance |

> This notebook spans **three swimlanes** — the most of any pattern. MDM sits in Governance, RAG in AI & Model Management, and the vector store in Analytical Data Management. That's why this is the capstone.

Our Plan B demo stitched 6 open-source tools together in Python. Plan A (IBM) provides this as a managed, governed, auditable platform. The compliance team cares about the latter. The data scientist loves the former. The architect bridges both.

In [19]:
print("📊 Reference Architecture Swimlane: Multiple (this notebook spans 3)")
print()
print("IBM Product Mapping:")
print("  Entity Resolution     → IBM Match 360 (probabilistic + deterministic matching)")
print("  Document Intelligence → Watson Discovery (document parsing + enrichment)")
print("  Embeddings            → watsonx.ai (slate.125m, granite-embedding)")
print("  Vector Store          → Elasticsearch in Watson Discovery")
print("  RAG Orchestration     → watsonx.ai RAG pattern (retrieval + generation)")
print("  Catalog & Lineage     → IBM Knowledge Catalog")
print()
print("Key differentiator:")
print("  Plan B = 6 open-source tools + Python glue code.")
print("  Plan A = managed platform with governance, audit trails, and SLAs.")
print("  The technology is similar. The difference is operational maturity.")

📊 Reference Architecture Swimlane: Multiple (this notebook spans 3)

IBM Product Mapping:
  Entity Resolution     → IBM Match 360 (probabilistic + deterministic matching)
  Document Intelligence → Watson Discovery (document parsing + enrichment)
  Embeddings            → watsonx.ai (slate.125m, granite-embedding)
  Vector Store          → Elasticsearch in Watson Discovery
  RAG Orchestration     → watsonx.ai RAG pattern (retrieval + generation)
  Catalog & Lineage     → IBM Knowledge Catalog

Key differentiator:
  Plan B = 6 open-source tools + Python glue code.
  Plan A = managed platform with governance, audit trails, and SLAs.
  The technology is similar. The difference is operational maturity.


---
## Section 7: BFSI Reality Check

Entity resolution is not optional for Canadian banks — OSFI Guideline B-8 requires that banks maintain a consolidated view of customer relationships for AML/KYC purposes. When a customer has a retail chequing account, a commercial mortgage, and a wealth management portfolio, the bank must know they are the same person. FINTRAC requires that Suspicious Transaction Reports include all known relationships, not just the one that triggered the alert. Without MDM, a $9,500 cash deposit at one branch and a $9,500 cash deposit at another branch on the same day — classic structuring — might never be linked because the transaction monitoring system sees two different customer IDs.

RAG over policy documents is the next frontier. Every major Canadian bank has hundreds of internal policy documents — AML, KYC, sanctions, PEP, fraud, privacy — that compliance officers must consult daily. Today, they search SharePoint by keyword and hope. Semantic search promises better: "What are our obligations when a customer is identified as a PEP and simultaneously triggers a sanctions match?" — a question that spans MTB-POL-005, MTB-POL-007, and MTB-POL-008. Our eval showed that naive RAG handles single-document lookups well but struggles with multi-document reasoning. Production RAG needs domain fine-tuning, multi-hop retrieval, and human-in-the-loop verification.

The banks that will succeed with AI-augmented compliance are the ones that treat it as an *assistant* — surfacing relevant policies, flagging risky customers, linking entities across systems — with a human compliance officer making the final decision. The ones that will fail are the ones that deploy unvalidated RAG pipelines to replace the compliance team. OSFI will not accept "the AI told us" as a defence in a supervisory review.

In [20]:
# ── Cleanup ───────────────────────────────────────────────────────────
con.close()

print("Notebook 6 complete.")
print()
print("What we demonstrated:")
print(f"  1. MDM entity resolution — {n_entities} entities unified across 4 source systems")
print(f"  2. RAG pipeline — {len(parsed_docs)} policy PDFs parsed, {len(all_chunks)} chunks indexed")
print("  3. Q2 answered — policy-driven customer identification (RAG + MDM)")
print(f"  4. Eval scoring — Recall@{k} on {len(eval_data)} Q&A pairs, single-doc vs multi-doc")
print()
print("Six patterns. One architecture.")
print("  NB1: Warehouse stores the structured data.")
print("  NB2: Lake stores the raw files.")
print("  NB3: Lakehouse adds ACID and time travel.")
print("  NB4: Virtualization federates access.")
print("  NB5: Mesh organizes domain ownership.")
print("  NB6: MDM + RAG adds intelligence — entity resolution and document understanding.")
print()
print("Together, these six patterns ARE the modern data architecture.")

Notebook 6 complete.

What we demonstrated:
  1. MDM entity resolution — 176 entities unified across 4 source systems
  2. RAG pipeline — 10 policy PDFs parsed, 637 chunks indexed
  3. Q2 answered — policy-driven customer identification (RAG + MDM)
  4. Eval scoring — Recall@5 on 30 Q&A pairs, single-doc vs multi-doc

Six patterns. One architecture.
  NB1: Warehouse stores the structured data.
  NB2: Lake stores the raw files.
  NB3: Lakehouse adds ACID and time travel.
  NB4: Virtualization federates access.
  NB5: Mesh organizes domain ownership.
  NB6: MDM + RAG adds intelligence — entity resolution and document understanding.

Together, these six patterns ARE the modern data architecture.


---

*Previous: [Notebook 5 — Data Mesh](05-data-mesh.ipynb)* | Series complete.